In [1]:
import pandas as pd
import numpy as np
import mysql.connector

In [2]:
# SQL Database Connection
def connection():
    return mysql.connector.connect(
    host="localhost",
    user="root",
    password="Sandeep@752081",
    database="inventory_db")

# Check if the connection is established
connection().is_connected()

True

In [3]:
db=connection()
cursor= db.cursor(dictionary= True)

In [4]:
cursor.execute("select count(supplier_id) as total_suppliers from suppliers")
cursor.fetchone()

{'total_suppliers': 50}

In [19]:
## function to fetch the basic details

def get_basic_info(cursor):
    """
    Retrieve summary inventory and supply chain metrics.

    Args:
        cursor (mysql.connector.cursor.MySQLCursorDict): Cursor object with dictionary=True.

    Returns:
        dict: Dictionary of metric labels and their values.
    """

    queries = {
        "Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

        "Total Products": "SELECT COUNT(*) AS count FROM products",

        "Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

        "Total Sale Value (Last 3 Months)": """
            SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) AS total_sale
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Sale'
              AND se.entry_date >= (
                  SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Total Restock Value (Last 3 Months)": """
            SELECT ROUND(SUM(se.change_quantity * p.price), 2) AS total_restock
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Restock'
              AND se.entry_date >= (
                  SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Below Reorder & No Pending Reorders": """
            SELECT COUNT(*) AS below_reorder
            FROM products p
            WHERE p.stock_quantity < p.reorder_level
              AND p.product_id NOT IN (
                  SELECT DISTINCT product_id FROM reorders WHERE status = 'Pending')
        """
    }

    results = {}
    for label, query in queries.items():
        cursor.execute(query)
        row = cursor.fetchone()
        # Since row is a dictionary, extract the single value by getting the first value in dict.values()
        results[label] = list(row.values())[0]

    return results

In [24]:
def get_additonal_tables(cursor):
    queries = {
        "Suppliers Contact Details": "SELECT supplier_name, contact_name, email, phone FROM suppliers",

        "Products with Supplier and Stock": """
            SELECT 
                p.product_name,
                s.supplier_name,
                p.stock_quantity,
                p.reorder_level
            FROM products p
            JOIN suppliers s ON p.supplier_id = s.supplier_id
            ORDER BY p.product_name ASC
        """,

        "Products Needing Reorder": """
            SELECT product_name, stock_quantity, reorder_level
            FROM products
            WHERE stock_quantity <= reorder_level
        """
    }

    tables = {}
    for label, query in queries.items():
        cursor.execute(query)
        tables[label] = cursor.fetchall()

    return tables

In [5]:
def add_manual_id(cursor, db, p_name,p_category, p_price, p_stock_quantity, p_reorder_level, p_supplier_id):
    proc_call= "call AddNewProductManualID(%s, %s, %s, %s, %s, %s)"
    params= (p_name, p_category, p_price, p_stock_quantity, p_reorder_level, p_supplier_id)
    cursor.execute(proc_call, params)
    db.commit()

In [12]:
def get_categories(cursor):
    cursor.execute("select distinct category from products order by category asc")
    rows= cursor.fetchall()
    return [i['category'] for i in rows]

get_categories(cursor)

['Clothing', 'Electronics', 'Furniture', 'Groceries', 'Toys']

In [15]:
def get_suppliers(cursor):
    cursor.execute("select supplier_id, supplier_name from suppliers order by supplier_name asc")
    rows= cursor.fetchall()
    return rows

get_suppliers(cursor)

[{'supplier_id': 1, 'supplier_name': 'Anderson-Thompson'},
 {'supplier_id': 48, 'supplier_name': 'Armstrong-Vance'},
 {'supplier_id': 24, 'supplier_name': 'Barker, White and Carson'},
 {'supplier_id': 25, 'supplier_name': 'Barrett Ltd'},
 {'supplier_id': 3, 'supplier_name': 'Baxter-Meadows'},
 {'supplier_id': 19, 'supplier_name': 'Charles Inc'},
 {'supplier_id': 41, 'supplier_name': 'Clark Group'},
 {'supplier_id': 23, 'supplier_name': 'Douglas Ltd'},
 {'supplier_id': 32, 'supplier_name': 'Elliott-Ayers'},
 {'supplier_id': 7, 'supplier_name': 'Evans Inc'},
 {'supplier_id': 27, 'supplier_name': 'Franklin, Kane and Price'},
 {'supplier_id': 15, 'supplier_name': 'Freeman-Gordon'},
 {'supplier_id': 13, 'supplier_name': 'Gallagher-Miller'},
 {'supplier_id': 17, 'supplier_name': 'Gomez PLC'},
 {'supplier_id': 40, 'supplier_name': 'Hall-Brown'},
 {'supplier_id': 38, 'supplier_name': 'Harris-Cummings'},
 {'supplier_id': 42, 'supplier_name': 'Henderson LLC'},
 {'supplier_id': 50, 'supplier_name